In [31]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain_community.vectorstores import FAISS
import os
from dotenv import load_dotenv

In [7]:
documents = [
    "Python is a versatile programming language used for web development and data science.",
    "Machine learning models require large amounts of training data to perform well.",
    "Neural networks are inspired by the structure of the human brain.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning is a subset of machine learning using multi-layered neural networks."
]

formatted_documents = [Document(page_content=document) for document in documents]

In [8]:
formatted_documents[0].page_content

'Python is a versatile programming language used for web development and data science.'

In [16]:
#Create splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 20,
    length_function = len
)

chunks = splitter.split_documents(formatted_documents)

print(f"Split {len(formatted_documents)} into {len(chunks)}")

Split 5 into 5


In [21]:
load_dotenv()
api_key = os.getenv("api_key")
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small",
                              api_key=api_key)

vector_store = FAISS.from_documents(chunks, embeddings)

query = "What is FAISS?"
results = vector_store.similarity_search(query, k=2)

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}: {doc.page_content}")



Result 1: Deep learning is a subset of machine learning using multi-layered neural networks.

Result 2: Machine learning models require large amounts of training data to perform well.


In [25]:
simple_llm = OpenAI(api_key=api_key, temperature=0.5)
simple_llm.invoke("What is today's weather")

'\n\nI am an AI and do not have access to real-time weather data. Please check your local weather forecast for accurate information.'

In [32]:
#create llm
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer using only the provided context."),
        ("human", "{question}\n\nContext:\n{context}")
    ]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})


rag_chain = (
    RunnableParallel(context = retriever | format_docs, question = RunnablePassthrough())
    |prompt
    |simple_llm
    |StrOutputParser()
)

In [34]:
response = rag_chain.invoke("What is langchain?")
print(response)



Langchain is not a specific term or concept in the context of natural language processing or deep learning. It is possible that it refers to a specific language processing tool or platform, but without more information it is not possible to provide a more specific answer.


In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder

chat_store = {}

In [ ]:
You are a professional virtual assistant representing business coach Sarah Chen.Given the details, you are required to prepare an email. Below are your roles:

1. Your response should be brief, factual and concise.
2. Your email should acknowledge their specific challenge, position Sarah as the
solution, and include a soft CTA to book a call.
3. Store the following(Note that Your response within the square brackets signifies LLM's reply):
Name: {{ $json.body.name }}
Email: {{ $json.body.email }}
Qualification Status: Qualified
Response: [Your response]

Remove place holders and replace with actual names, position and possibly contact information.

Phone number: 0810001000
Email: sarahchen@gmail.com



In [ ]:
Write an appriopriate email based on the given system instructions. 

Name: {{ $json.body.name }}
Email: {{ $json.body.email }}
Business Type: {{ $json.body.business_type }}
Monthly Revenue: ${{ $json.body.monthly_revenue }}
Biggest Challenge: ${{ $json.body.biggest_challenge }}
